# スコアが最大の依存構造木を求める (最大全域木)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/08_parsing/mst_decoding.ipynb)

NLP クックブック 08-13 ／ 構文解析

「どの語がどの語に係りそうか」のスコアが全ペアについて与えられたとき、全体として最も良い木を選びたい。グラフに基づく解析の復号を理解したい。

In [1]:
import numpy as np

words = ["ROOT", "John", "saw", "Mary"]
# score[h, d] = 語 h が語 d の主辞である、というスコア (実際のパーザではニューラルネットが出力する)
score = np.array([
    [-np.inf,  9, 10,  9],    # ROOT → John / saw / Mary
    [-np.inf, -np.inf, 20,  3],    # John →
    [-np.inf, 30, -np.inf, 30],    # saw  →
    [-np.inf, 11,  0, -np.inf],    # Mary →
])

# 貪欲法: 各語について、スコアが最大の主辞を独立に選ぶ
greedy = {words[d]: words[int(score[:, d].argmax())] for d in range(1, len(words))}
print("貪欲法:", greedy)   # John → saw → John の循環ができて、木にならない

貪欲法: {'John': 'saw', 'saw': 'John', 'Mary': 'saw'}


In [2]:
import networkx as nx

def mst(score):
    """Chu-Liu/Edmonds のアルゴリズムで、ROOT (0 番) を根とする最大全域有向木を求める"""
    G = nx.DiGraph()
    n = len(score)
    for h in range(n):
        for d in range(1, n):
            if h != d and np.isfinite(score[h, d]):
                G.add_edge(h, d, weight=float(score[h, d]))
    tree = nx.maximum_spanning_arborescence(G)
    heads = {d: h for h, d in tree.edges}
    return [heads[d] for d in range(1, n)], sum(score[heads[d], d] for d in range(1, n))

heads, total = mst(score)
print("最大全域木:", {words[d]: words[h] for d, h in enumerate(heads, 1)}, "合計スコア:", total)

最大全域木: {'John': 'saw', 'saw': 'ROOT', 'Mary': 'saw'} 合計スコア: 70.0


In [3]:
# 総当たりで確かめる: すべての主辞の割り当てのうち、木になっているものの中で最大のもの
from itertools import product

def is_tree(heads):
    for d in range(1, len(heads) + 1):
        seen, cur = set(), d
        while cur != 0:
            if cur in seen:
                return False
            seen.add(cur); cur = heads[cur - 1]
    return True

best = max((h for h in product(range(4), repeat=3) if all(h[d - 1] != d for d in range(1, 4)) and is_tree(h)),
           key=lambda h: sum(score[h[d - 1], d] for d in range(1, 4)))
print("総当たり:", list(best), "一致:", list(best) == heads)

総当たり: [2, 0, 2] 一致: True


- グラフに基づく解析は「係り受けのスコアを付ける」と「スコア最大の木を探す」の 2 段階。前者を biaffine 注意機構で計算するのが、Stanza や GiNZA (Transformer 版) などで使われている現在の標準的な方式。
- 最大全域木は交差する係り受けも扱える。交差を許さない場合は Eisner のアルゴリズム (動的計画法、O(n³)) を使う。
- ROOT から出る辺が複数になりうる点に注意。「ROOT の子は 1 つ」という UD の制約を守らせるには追加の処理が要る。
- 遷移型は局所的な判断の積み重ねで速く (O(n))、グラフ型は文全体で最適化する。長距離の係り受けはグラフ型が有利とされる。